# Sentiment Analysis Using Traditional Machine Learning
**PUSL 3205 – Large Language Models and Generative AI | Lab 1**

Dataset: [UCI Sentiment Labelled Sentences](https://archive.ics.uci.edu/dataset/331/sentiment+labelled+sentences), which has 3,000 sentences from Amazon, IMDb and Yelp, each labelled positive (1) or negative (0).

## Task 1: Load the Dataset

In [4]:
import csv
import pandas as pd
import numpy as np

In [5]:
files = {
    "amazon": "../data/amazon_cells_labelled.txt",
    "imdb":   "../data/imdb_labelled.txt",
    "yelp":   "../data/yelp_labelled.txt",
}

frames = []
for source, path in files.items():
    part = pd.read_csv(
        path,
        sep="\t",
        header=None,
        names=["text", "label"],
        quoting=csv.QUOTE_NONE,  # IMDb reviews contain stray quote marks
    )
    part["source"] = source
    frames.append(part)

df = pd.concat(frames, ignore_index=True)
df.head()

,text,label,source
0,So there is no way for me to plug it in here i...,0,amazon
1,"Good case, Excellent value.",1,amazon
2,Great for the jawbone.,1,amazon
3,Tied to charger for conversations lasting more...,0,amazon
4,The mic is great.,1,amazon


> **Note:** `quoting=csv.QUOTE_NONE` is required. Without it, pandas treats the `"` characters inside some IMDb reviews as quoted fields and silently loads only 748 of the 1,000 IMDb rows.

In [6]:
print("Shape:", df.shape)
print("\nRows per source:\n", df["source"].value_counts())
print("\nLabel balance:\n", df["label"].value_counts())
print("\nMissing values:\n", df.isnull().sum())

Shape: (3000, 3)

Rows per source:
 source
amazon    1000
imdb      1000
yelp      1000
Name: count, dtype: int64

Label balance:
 label
0    1500
1    1500
Name: count, dtype: int64

Missing values:
 text      0
label     0
source    0
dtype: int64


## Task 2: Text Preprocessing
Steps applied to every sentence:
1. Convert to lowercase
2. Expand negative contractions (`don't` → `do not`) so that negation survives punctuation removal
3. Remove punctuation and special characters (letters and digits are kept)
4. Remove extra whitespace

In [7]:
import re

def clean_text(text):
    # 1. Lowercase
    text = text.lower()

    # 2. Expand negative contractions (irregular ones first)
    text = re.sub(r"won't", "will not", text)
    text = re.sub(r"can't", "can not", text)
    text = re.sub(r"n't\b", " not", text)

    # 3. Remove punctuation / special characters (keep letters, digits, spaces)
    text = re.sub(r"[^a-z0-9\s]", " ", text)

    # 4. Remove extra whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text

# Quick test
clean_text("I DIDN'T like it... at ALL!!!   Won't buy again.")

'i did not like it at all will not buy again'

In [8]:
df["clean_text"] = df["text"].apply(clean_text)
df[["text", "clean_text", "label"]].head(10)

,text,clean_text,label
0,So there is no way for me to plug it in here i...,so there is no way for me to plug it in here i...,0
1,"Good case, Excellent value.",good case excellent value,1
2,Great for the jawbone.,great for the jawbone,1
3,Tied to charger for conversations lasting more...,tied to charger for conversations lasting more...,0
4,The mic is great.,the mic is great,1
5,I have to jiggle the plug to get it to line up...,i have to jiggle the plug to get it to line up...,0
6,If you have several dozen or several hundred c...,if you have several dozen or several hundred c...,0
7,If you are Razr owner...you must have this!,if you are razr owner you must have this,1
8,"Needless to say, I wasted my money.",needless to say i wasted my money,0
9,What a waste of money and time!.,what a waste of money and time,0


In [9]:
# Side-by-side examples
comparison = df[["text", "clean_text"]].sample(8, random_state=42)
pd.set_option("display.max_colwidth", None)
display(comparison)

# Before vs after statistics
stats = pd.DataFrame({
    "Original": [
        df["text"].str.len().mean(),
        df["text"].str.split().str.len().mean(),
        df["text"].str.count(r"[^\w\s]").sum(),
    ],
    "Cleaned": [
        df["clean_text"].str.len().mean(),
        df["clean_text"].str.split().str.len().mean(),
        df["clean_text"].str.count(r"[^\w\s]").sum(),
    ],
}, index=["Avg characters", "Avg words", "Total punctuation marks"])

stats.round(2)

,text,clean_text
1801,Avoid at ALL costs!,avoid at all costs
1190,"Garbo, who showed right off the bat that her talents could carry over from the silent era (I wanted to see some of her silent work, but Netflix doesn't seem to be stocking them.",garbo who showed right off the bat that her talents could carry over from the silent era i wanted to see some of her silent work but netflix does not seem to be stocking them
1817,You will leave the theater wanting to go out and dance under the stars.,you will leave the theater wanting to go out and dance under the stars
251,O my gosh the best phone I have ever had.,o my gosh the best phone i have ever had
2505,I would not recommend this place.,i would not recommend this place
1117,"Then scene where they debated whether or not to sack the trumpeter (who falsely was accused for the murder) is pure horror, really stupid.",then scene where they debated whether or not to sack the trumpeter who falsely was accused for the murder is pure horror really stupid
1411,Her role was played well.,her role was played well
2113,"Nice, spicy and tender.",nice spicy and tender


,Original,Cleaned
Avg characters,65.27,62.86
Avg words,11.83,12.13
Total punctuation marks,6465.00,0.00


### Observations
- Average sentence length dropped from ~65 to ~63 characters after cleaning.
- All punctuation was removed, while words and numbers were kept.
- Negations such as "didn't" became "did not", so the word **not** remains as a separate token.
- Numeric ratings such as "10/10" become "10 10" instead of disappearing.